# Saving module configurations & re-creating pipeline from config file

### Saving interactively added module configurations to a BioPAP config file

In [ ]:
# Make sure you have the most recent version installed.
#%pip install -e ..

In [ ]:
from biopap.modules.anonymize import Anonymize
from biopap.modules.artifact_reject import ArtifactReject
from biopap.modules.baseline_correction import BaselineCorrection
from biopap.modules.bids import BIDSConversion
from biopap.modules.blink_detection import BlinkDetection
from biopap.modules.downsample import Downsample
from biopap.modules.eog_channels import EOGChannels
from biopap.modules.iir_filter import IIRFilter
from biopap.modules.load_data import LoadData
from biopap.modules.plot import Plot, PlotTypes
from biopap.modules.rereference import Rereference
from biopap.modules.segmentation import Segmentation
from biopap.modules.save_output import SaveOutput
from biopap.modules.set_channel_type_and_montage import SetChannelTypeAndMontage

from biopap.pipeline import Pipeline
from biopap.modules.helpers import save_modules_config

In [ ]:
# Have a data file and load it
data_file = '/my/data/ABC1234.bdf'

load = LoadData(file_path=data_file)

# Add the modules you want to use, adding and testing the configuration for
# each until you are satisfied that each module's settings are correct.

ano = Anonymize(
    raw=load.raw,
    daysback=100,
    experimenter='J. Jansen',
    participant_id='AB00123')

In [ ]:
# You can use both the standard variables - in this case `ref_channels` -, as well as additional
# parameters to be passed on the the underlying mne-function, like `ch_type` in the example below.

ref = Rereference(
    raw=ano.processed,
    ref_channels=['EXG1', 'EXG2'],
    ch_type='auto')

In [ ]:
plt = Plot(
    raw=ano.processed,
    save_folder='./',
    save_postfix='IRR')

eog = EOGChannels(
    raw=ref.processed,
    ref_channels=['hEOG', 'vEOG'],
    anode_channels=['EXG3','EXG5'],
    cathode_channels=['EXG4','EXG6'])

channel_types = {
    'misc': ['EXG1','EXG2','EXG3','EXG4','EXG5','EXG6','EXG7','EXG8','Erg1','Erg2','Plet'],
    'gsr': ['GSR1', 'GSR2'],
    'resp': ['Resp'],
    'temperature': ['Temp']
}

montage = SetChannelTypeAndMontage(
    raw=eog.processed,
    channel_types=channel_types,
    mne_standard_montage='biosemi64'
)

flt = IIRFilter(
    raw=montage.processed,
    l_freq=0.1,
    h_freq=35)

dwn = Downsample(
    raw=flt.processed,
    sfreq=500)  # Downsample to 500 Hz

blink = BlinkDetection(raw=dwn.processed)

In [ ]:
# Make sure when you have multiple instances of the same module
# in your pipeline to give each instance a different name (`plt` vs `plt2`).

plt2 = Plot(
    raw=blink.processed,
    title="Show blink annotations")

In [ ]:
seg = Segmentation(
    raw=blink.processed,  
    stim_channel='Status',  
    start=-0.2,  
    end=0.8,     
    baseline=(None, 0),
    reference_markers={'range': '1-69'})

artifact = ArtifactReject(
    raw=seg.processed,
    mode='interpolation',
    max_gradient_uv=50,
    max_amplitude_uv=75,
    min_activity_uv=0.5,
    interval_ms=100)

baseline = BaselineCorrection(
    raw=artifact.processed,
    baseline=(-0.2, 0))

pre_ica_save = SaveOutput(
    raw=baseline.processed,
    export_formats='fif',
    output_file='./partially_processed.fif',
    overwrite=True,
    intermediate='_before_ICA')

plt3 = Plot(
    raw=pre_ica_save.processed,
    title="Preprocessed data")

In [ ]:
save_output = SaveOutput(
    raw=pre_ica_save.processed,
    export_formats='fif',
    output_file='./processed_data-epo.fif',
    intermediate=False,
    overwrite=True)

Once you are done with adding modules and editing their configuration, the settings for the complete pipeline can be saved to a configuration file by calling the `save_modules_config` helper function.

The order of the modules is important, as this will also be the order in which the modules will be added when a new pipeline is created from the saved configuration file.

Note that you only need to pass the function a list of the modules that you actually want added to the config. For instance, the example below omits `plt2`, the second plotting step.

Also note that it is not necessary to save the settings from the `Load` module, as that module is also integrated in the `Pipeline` class.

In [ ]:
cfg_file = './config.yaml'
save_modules_config(
    modules=[ano, ref, plt, eog, flt, dwn, seg, artifact, baseline, pre_ica_save, ica_result, plt3, save_output], 
    config_path=cfg_file
)

Please note: besides saving a pipeline configuration from individual modules as shown above, you can also first manually create the pipeline, add all the steps and their configuration, and call the pipeline's `to_config()` method.

# Re-creating pipeline from config file

By loading a saved config file into a new `Pipeline`-instance, you automatically create the complete pipeline with all the specified settings.

In [ ]:
cfg_file = './config.yaml'

new_pipe = Pipeline.from_config(
    config_file=cfg_file,
    input_file=data_file,
    subject_id='ABC123'
)

new_pipe.inspect()

## A word on subject_id

The subject ID can be used to differentiate between outputs for different users when running the pipeline in batches. When you use the string `{subject_id}` in the value of any string-type config variable, it will be automatically replaced by the value of subject_id. So, for instance, by defining a pipeline-step thus:

```python
SaveOutput(
    raw=baseline.processed,
    export_formats='fif',
    output_file='./partially_processed_{subject_id}.fif',
    overwrite=True,
    intermediate='_before_ICA')
```
running the pipeline with `subject_id="A123"` will result in an outputfile `partially_processed_A123.fif`.

If you do not specify a subject_id when creating the pipeline, the program will use the input file's name without the suffix. It will first try to read this from the 'filenames' attribute from the file's metadata, and if that is empty, use the actual file's path.